## Download Protein Datasets and Models

This section parses the repository asset index (`urls.txt`) and downloads all protein-related datasets and pretrained models into local directories.

### Imports

In [1]:
from pathlib import Path   # File system path handling
import requests            # HTTP requests for downloading files
from tqdm import tqdm      # Progress bar for downloads

### Configure Paths

In [2]:
def ensure_urls_file(urls_file: Path, base_url: str) -> Path:
    """Ensure urls.txt exists locally; download if missing."""

    # Check if urls.txt is already available locally
    if not urls_file.exists():
        url = f"{base_url}urls.txt"  # Full remote path to urls.txt

        # Download file from remote server
        response = requests.get(url, timeout=60)
        response.raise_for_status()  # Raise error if request failed

        # Save content to local file
        urls_file.write_text(response.text, encoding="utf-8")
        print(f"Downloaded: {urls_file}")
    else:
        print(f"Already exists: {urls_file}")

    return urls_file  # Return validated local path

In [3]:
# Base URL hosting all downloadable assets (datasets and pretrained models)
BASE_URL = "https://assets.deep-learning-for-biology.com/"

URLS_FILE = Path("./urls.txt")                      # Path to local urls.txt
URLS_FILE = ensure_urls_file(URLS_FILE, BASE_URL)   # Ensure urls.txt exists locally

DATASETS_DIR = Path("./datasets")  # Local dataset storage
MODELS_DIR = Path("./models")      # Local model storage

# Create directories if they do not exist
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

Downloaded: urls.txt


### Parse Asset Index

In [4]:
# Read asset index file
lines = URLS_FILE.read_text().splitlines()

protein_files = []

# Filter protein datasets and model files
for line in lines:
    parts = line.strip().split(maxsplit=1)

    if len(parts) != 2:
        continue

    size, relative_path = parts
    size = int(size)

    # Dataset files → flat structure
    if relative_path.startswith("proteins/datasets/"):
        output_path = DATASETS_DIR / Path(relative_path).name
        protein_files.append((size, relative_path, output_path))

    # Model files → preserve directory hierarchy
    elif relative_path.startswith("proteins/models/"):
        model_relative_path = Path(relative_path).relative_to("proteins/models")
        output_path = MODELS_DIR / model_relative_path
        protein_files.append((size, relative_path, output_path))

print(f"Files to download: {len(protein_files)}")

Files to download: 22


### Download Utility

In [5]:
# Download a single file with progress tracking
def download_file(relative_path, output_path, chunk_size=8192):
    url = BASE_URL + relative_path

    # Ensure parent directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip if file already exists
    if output_path.exists():
        print(f"Skipping existing file: {output_path}")
        return

    # Stream download with progress bar
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()

        total_size = int(response.headers.get("content-length", 0))

        with open(output_path, "wb") as file, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            desc=output_path.name
        ) as progress_bar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress_bar.update(len(chunk))

### Execute Download

In [6]:
# Download all selected protein datasets and models
for size, relative_path, output_path in protein_files:
    download_file(relative_path, output_path)

go_term_descriptions.csv: 100%|██████████| 2.10M/2.10M [00:00<00:00, 55.3MB/s]
protein_dataset_test_facebook_esm2_t30_150M_UR50D.feather: 100%|██████████| 3.88M/3.88M [00:00<00:00, 68.5MB/s]
protein_dataset_train_facebook_esm2_t30_150M_UR50D.feather: 100%|██████████| 10.8M/10.8M [00:00<00:00, 85.7MB/s]
protein_dataset_valid_facebook_esm2_t30_150M_UR50D.feather: 100%|██████████| 3.87M/3.87M [00:00<00:00, 76.9MB/s]
sequence_df_cco.csv: 100%|██████████| 212M/212M [00:01<00:00, 114MB/s]  
sequence_df_mfo.csv: 100%|██████████| 102M/102M [00:00<00:00, 109MB/s]  
train_sequences.fasta: 100%|██████████| 97.3M/97.3M [00:00<00:00, 105MB/s] 
train_taxonomy.tsv.zip: 100%|██████████| 763k/763k [00:00<00:00, 33.9MB/s]
train_terms.tsv.zip: 100%|██████████| 19.4M/19.4M [00:00<00:00, 104MB/s] 
_CHECKPOINT_METADATA: 100%|██████████| 371/371 [00:00<?, ?B/s] 
metadata: 100%|██████████| 15.0k/15.0k [00:00<00:00, 15.7MB/s]
_METADATA: 100%|██████████| 6.18k/6.18k [00:00<00:00, 6.33MB/s]
_sharding: 100%|█████

All protein datasets and models are now stored locally:

- `./datasets/` → tabular datasets and sequence files  
- `./models/` → pretrained model checkpoints  

This setup enables fully local, reproducible workflows without relying on external provisioning scripts.